In [32]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Set random seed for reproducibility
np.random.seed(7)

# Simulation parameters
population_mean = 100
population_std = 15
sample_size = 30
num_samples = 20

# Confidence level (only 90%)
confidence_level = 0.90

# Generate random samples from the population
samples = [np.random.normal(population_mean, population_std, sample_size) for _ in range(num_samples)]

# Calculate sample statistics
means = [np.mean(sample) for sample in samples]                                    # Sample means
std_errors = [np.std(sample, ddof=1) / np.sqrt(sample_size) for sample in samples] # Standard errors

# Critical z-score for 90% CI
z_score = norm.ppf(1 - (1 - confidence_level) / 2)

# Confidence intervals: mean ± z * standard_error
conf_intervals = [(m - z_score * se, m + z_score * se) for m, se in zip(means, std_errors)]

# Plot population distribution as reference
x = np.linspace(population_mean - 3*population_std, population_mean + 3*population_std, 1000)
y = norm.pdf(x, population_mean, population_std)
plt.plot(x, y, color='black', lw=2, label='Population Distribution')

# Scale the distribution curve to fit nicely above the intervals
y = y / np.max(y) * 0.2 * num_samples
plt.fill_between(x, 0, y, color='lightgray', alpha=0.6)

# Plot each confidence interval as horizontal line
for i, (mean, (ci_lower, ci_upper)) in enumerate(zip(means, conf_intervals)):
    color = 'blue' if ci_lower <= population_mean <= ci_upper else 'red'
    plt.plot([ci_lower, ci_upper], [i, i], color=color, lw=2)   # Interval line
    plt.scatter(mean, i, color='black', marker='D')             # Sample mean

# True population mean reference line
plt.axvline(population_mean, color='gray', linestyle='--', label='True Population Mean')

# Format plot
plt.title(f'Confidence Intervals (Confidence Level: {confidence_level * 100:.0f}%)')
plt.xlabel('Value')
plt.ylabel('Sample Index')
plt.legend()
plt.tight_layout()
plt.savefig("confidence_intervals.png", dpi=300, bbox_inches="tight")
plt.close()

from google.colab import files
files.download('confidence_intervals.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [74]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
import pandas as pd

# Set random seed for reproducibility
np.random.seed(7)

# Hypothesis Test Class
class HypothesisTest:
    """
    Perform and visualize one-sample t-tests for population mean.
    Tests H₀: μ = null_value vs H₁: μ ≠ null_value (or >, <)
    """
    def __init__(self, data, null_value, alternative='two-sided', alpha=0.05):
        self.data = np.array(data)
        self.n = len(data)
        self.null_value = null_value
        self.alternative = alternative    # 'two-sided', 'greater', or 'less'
        self.alpha = alpha

        # Calculate sample statistics
        self.sample_mean = np.mean(data)
        self.sample_std = np.std(data, ddof=1)
        self.se = self.sample_std / np.sqrt(self.n)

    def perform_test(self):
        """Execute the t-test and calculate p-value and critical values."""
        self.t_stat = (self.sample_mean - self.null_value) / self.se
        df = self.n - 1

        if self.alternative == 'two-sided':
            self.p_value = 2 * t.cdf(-abs(self.t_stat), df)
            self.critical_value = t.ppf(1 - self.alpha/2, df)
        elif self.alternative == 'greater':
            self.p_value = 1 - t.cdf(self.t_stat, df)
            self.critical_value = t.ppf(1 - self.alpha, df)
        else:  # 'less'
            self.p_value = t.cdf(self.t_stat, df)
            self.critical_value = -t.ppf(1 - self.alpha, df)

        self.reject_null = self.p_value < self.alpha

        return {
            't_statistic': self.t_stat,
            'p_value': self.p_value,
            'critical_value': self.critical_value,
            'reject_null': self.reject_null,
            'sample_mean': self.sample_mean,
            'standard_error': self.se
        }

    def visualize(self, filename=None):
        """Create two plots and save to file if filename is given."""
        result = self.perform_test()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left plot: t-distribution
        x = np.linspace(-4, 4, 1000)
        y = t.pdf(x, self.n - 1)
        axes[0].plot(x, y, 'b-', linewidth=2, label='t-distribution')

        # Shade rejection region
        axes[0].fill_between(
            x, y,
            where=(x <= -abs(self.critical_value)) | (x >= abs(self.critical_value)),
            alpha=0.3, color='red', label=f'Rejection region (α={self.alpha})'
        )

        # Mark observed t-statistic
        axes[0].axvline(self.t_stat, color='green', linestyle='--', linewidth=2,
                        label=f't-stat = {self.t_stat:.3f}')

        if self.alternative == 'two-sided':
            axes[0].axvline(self.critical_value, color='red', linestyle=':', linewidth=1)
            axes[0].axvline(-self.critical_value, color='red', linestyle=':', linewidth=1)

        axes[0].set_xlabel('t-value')
        axes[0].set_ylabel('Probability Density')
        axes[0].set_title('Test Statistic on t-Distribution')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Right plot: Sample histogram
        axes[1].hist(self.data, bins=20, alpha=0.7, color='skyblue',
                     edgecolor='black', density=True)
        axes[1].axvline(self.sample_mean, color='green', linestyle='--', linewidth=2,
                        label=f'Sample mean = {self.sample_mean:.2f}')
        axes[1].axvline(self.null_value, color='red', linestyle='--', linewidth=2,
                        label=f'Null value = {self.null_value}')
        axes[1].set_xlabel('Value')
        axes[1].set_ylabel('Density')
        axes[1].set_title('Sample Distribution vs. Null Hypothesis')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.suptitle(
            f'Hypothesis Test: p-value = {result["p_value"]:.4f}, ' +
            ('Reject H₀' if result['reject_null'] else 'Fail to reject H₀'),
            fontsize=14, y=1.02
        )
        plt.tight_layout()

        if filename:
            plt.savefig(filename, dpi=300, bbox_inches='tight')
        else:
            plt.show()

        plt.close()
        return result


# Example Usage
print("Hypothesis Test Example: Temperature Average")
print("=" * 30)
print("H₀: μ = 30 (null hypothesis)")
print("H₁: μ ≠ 30 (alternative hypothesis)")
print("α = 0.05\n")

# Two datasets
temp_avg1 = np.random.normal(30.7, 5, 30)     # small dataset
temp_avg2 = np.random.normal(30.7, 5, 800)   # large dataset

# Run tests
tester_small = HypothesisTest(temp_avg1, null_value=30, alternative='two-sided', alpha=0.05)
result_small = tester_small.visualize("hypothesis_test_small.png")

tester_large = HypothesisTest(temp_avg2, null_value=30, alternative='two-sided', alpha=0.05)
result_large = tester_large.visualize("hypothesis_test_large.png")

# Save results to CSV
df_results = pd.DataFrame([
    {"Sample Size": tester_small.n, **result_small},
    {"Sample Size": tester_large.n, **result_large}
])
df_results.to_csv("hypothesis_test_results.csv", index=False)

from google.colab import files
files.download("hypothesis_test_results.csv")
files.download("hypothesis_test_small.png")
files.download("hypothesis_test_large.png")

Hypothesis Test Example: Temperature Average
H₀: μ = 30 (null hypothesis)
H₁: μ ≠ 30 (alternative hypothesis)
α = 0.05



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mannwhitneyu
import pandas as pd

# Functions
class MannWhitneyTest:
    def __init__(self, group1, group2, alpha=0.05):
        self.group1 = np.array(group1, dtype=float)
        self.group2 = np.array(group2, dtype=float)
        self.alpha = alpha
        self.n1 = len(self.group1)
        self.n2 = len(self.group2)

        # Precompute ranks
        self._combined = np.concatenate([self.group1, self.group2])
        self._ranks = stats.rankdata(self._combined, method="average")
        self._ranks1 = self._ranks[:self.n1]
        self._ranks2 = self._ranks[self.n1:]

    def perform_test(self):
        statistic, p_value = mannwhitneyu(
            self.group1, self.group2, alternative='two-sided', method='auto'
        )
        return {
            'U_statistic': float(statistic),
            'p_value': float(p_value),
            'median1': float(np.median(self.group1)),
            'median2': float(np.median(self.group2)),
            'mean_rank1': float(np.mean(self._ranks1)),
            'mean_rank2': float(np.mean(self._ranks2)),
            'reject_null': p_value < self.alpha
        }

    def visualize(self, filename="mannwhitney_plot.png"):
        result = self.perform_test()
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Panel 1: Rank distribution
        axes[0].scatter(np.ones(self.n1), self._ranks1, alpha=0.6, s=50, label='Model A')
        axes[0].scatter(np.ones(self.n2)*2, self._ranks2, alpha=0.6, s=50, label='Model B')
        axes[0].axhline(result['mean_rank1'], linestyle='--', alpha=0.5, color='C0')
        axes[0].axhline(result['mean_rank2'], linestyle='--', alpha=0.5, color='C1')
        axes[0].set_xticks([1, 2])
        axes[0].set_xticklabels(['Model A', 'Model B'])
        axes[0].set_ylabel('Rank')
        axes[0].set_title('Rank Distribution')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Panel 2: Box plots
        axes[1].boxplot([self.group1, self.group2], labels=['Model A', 'Model B'])
        axes[1].set_ylabel('Value')
        axes[1].set_title('Box Plots')
        axes[1].grid(True, alpha=0.3)

        # Panel 3: Violin plots
        axes[2].violinplot([self.group1, self.group2], positions=[1, 2],
                           widths=0.6, showmeans=True, showmedians=True)
        axes[2].set_xticks([1, 2])
        axes[2].set_xticklabels(['Model A', 'Model B'])
        axes[2].set_ylabel('Value')
        axes[2].set_title('Distribution (Violin)')
        axes[2].grid(True, alpha=0.3)

        plt.suptitle(
            f"Mann-Whitney U: p = {result['p_value']:.4f}  —  "
            f"{'Reject H₀' if result['reject_null'] else 'Fail to reject H₀'}",
            fontsize=14, y=1.02
        )
        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        return result, filename


# Data
model_a_scores = [0.92, 0.81, 0.65, 0.48]
model_b_scores = [0.91, 0.85, 0.67, 0.51, 0.45]

mw_test = MannWhitneyTest(model_a_scores, model_b_scores)
mw_result, plot_file = mw_test.visualize("mannwhitney_models.png")

# Table
df_results = pd.DataFrame([mw_result])
csv_file = "mannwhitney_results.csv"
df_results.to_csv(csv_file, index=False)

# Download
from google.colab import files
files.download(csv_file)
files.download(plot_file)

/tmp/ipython-input-2787426169.py:53: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1].boxplot([self.group1, self.group2], labels=['Model A', 'Model B'])


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>